# Molecular Networking Tutorial

This notebook introduces **molecular networking** in CoreMS: building networks of MS2 spectra using **entropy similarity** and optional **cosine** scores, in **open** or **neutral-loss** (or identity) search modes.

### What you will do
1. Load a small public MSP library (`tests/tests_data/lcms/test_db.msp`)
2. Build a FlashEntropy spectral library
3. Build a **query-only** network from mock experimental spectra
4. Build a **query + library** network from noisy copies of library spectra
5. Inspect edges / statistics and (optionally) plot with `corems[networking]`

### Requirements
- CoreMS installed from this repo (`pip install -e .`)
- Optional visualization: `pip install "corems[networking]"` (`networkx`, `ipysigma`)

This notebook uses **only repo fixtures** (no Thermo RAW, no private paths). For LC-MS mass features with MS2, see `LCMS_Tutorial.ipynb` and `MolecularNetwork.prepare_query_spectra_from_lcms_object`.


In [1]:
## Imports and paths
from pathlib import Path

import numpy as np

from corems.molecular_id.search.database_interfaces import MSPInterface
from corems.molecular_networking import MolecularNetwork, SimilarityMatrix

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "tests" / "tests_data" / "lcms" / "test_db.msp").is_file():
    # Allow running from examples/notebooks/
    if (REPO_ROOT.parent.parent / "tests" / "tests_data" / "lcms" / "test_db.msp").is_file():
        REPO_ROOT = REPO_ROOT.parent.parent

MSP_FILE = REPO_ROOT / "tests" / "tests_data" / "lcms" / "test_db.msp"
OUT_DIR = REPO_ROOT / "temp.corems" / "molecular_networking_tutorial"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MSP: {MSP_FILE} (exists={MSP_FILE.is_file()})")
print(f"Output: {OUT_DIR}")
assert MSP_FILE.is_file(), f"Missing fixture MSP: {MSP_FILE}"


MSP: /Users/heal742/LOCAL/corems_dev/corems/tests/tests_data/lcms/test_db.msp (exists=True)
Output: /Users/heal742/LOCAL/corems_dev/corems/temp.corems/molecular_networking_tutorial


## 1. Build a FlashEntropy library from the test MSP

We use the same FlashEntropy kwargs pattern as the LC-MS tutorials. `min_ms2_difference_in_da` should be **2×** `max_ms2_tolerance_in_da`.


In [2]:
FE_KWARGS = {
    "normalize_intensity": True,
    "min_ms2_difference_in_da": 0.02,
    "max_ms2_tolerance_in_da": 0.01,
    "max_indexed_mz": 3000,
    "precursor_ions_removal_da": None,
    "noise_threshold": 0,
}

msp = MSPInterface(file_path=str(MSP_FILE))
fe_lib, metabolite_metadata = msp.get_metabolomics_spectra_library(
    polarity="negative",
    format="flashentropy",
    normalize=True,
    fe_kwargs=FE_KWARGS,
)
print(f"Library spectra (negative): {len(msp._data_frame[msp._data_frame.ionmode == 'negative'])}")
print(f"Metabolite metadata entries: {len(metabolite_metadata)}")
print(f"FlashEntropy type: {type(fe_lib).__name__}")


Library spectra (negative): 5
Metabolite metadata entries: 5
FlashEntropy type: FlashEntropySearch


## 2. Query-only network (no reference library)

Stage 1 only: pairwise similarities among experimental (mock) spectra. Useful when you have MS2 from a sample but no spectral library yet.


In [3]:
class MockSpectrum:
    """Minimal spectrum object for SimilarityEngine (mz_exp + abundance)."""

    def __init__(self, mz_exp, abundance, name=None):
        self.mz_exp = np.asarray(mz_exp, dtype=float)
        self.abundance = np.asarray(abundance, dtype=float)
        self.name = name


base_mz = np.array([100.0, 150.0, 200.0, 250.0, 280.0], dtype=float)
base_ab = np.array([1.0, 0.8, 0.6, 0.4, 0.2], dtype=float)

query_spectra = [
    MockSpectrum(base_mz, base_ab, name="EXP_0"),
    MockSpectrum(base_mz + 0.0001, base_ab, name="EXP_1_near_duplicate"),
    MockSpectrum(np.array([120.0, 180.0, 220.0]), np.array([1.0, 0.5, 0.3]), name="EXP_2_different"),
]
query_ids = ["q0", "q1", "q2"]
query_precursor_mzs = [300.0, 300.0, 250.0]

mn_q = MolecularNetwork(
    fe_lib=None,
    search_type="open",
    additional_similarities=["cosine"],
    similarity_thresholds={"entropy_similarity": 0.1, "cosine": 0.1},
    use_parallel=False,
)
mn_q.run_query_vs_query_only(
    query_spectra, query_ids, query_precursor_mzs=query_precursor_mzs
)

edges_q = mn_q.get_network_edges(metric="entropy_similarity")
stats_q = mn_q.get_network_stats(metric="entropy_similarity")
print("Query-only edges (entropy):")
for a, b, s in sorted(edges_q, key=lambda x: -x[2]):
    print(f"  {a} — {b}: {s:.4f}")
print("Stats:", stats_q)


Query-only edges (entropy):
  q0 — q1: 1.0000
  q0 — q2: 1.0000
  q1 — q2: 1.0000
Stats: {'metric': 'entropy_similarity', 'threshold': 0.1, 'n_nodes': 3, 'n_edges': 3, 'avg_degree': 2.0, 'density': 1.0}


## 3. Query + library network

Create noisy “experimental” queries from the MSP library rows, then run the full staged search:
- Stage 1: query–query  
- Stage 2: query–library  
- Stage 3 (optional): library–library among library hits  

We use `search_type="open"` here; try `"neutral_loss"` or `"identity"` with appropriate precursor m/z values.


In [4]:
df = msp._data_frame
# Prefer negative-mode rows if present
if "ionmode" in df.columns:
    df_use = df[df["ionmode"].astype(str).str.lower() == "negative"].copy()
    if df_use.empty:
        df_use = df.copy()
else:
    df_use = df.copy()

n_queries = min(4, len(df_use))
lib_spectra = []
lib_ids = []
lib_pmz = []

for i in range(n_queries):
    row = df_use.iloc[i]
    peaks = np.asarray(row.peaks, dtype=float)
    if peaks.size == 0:
        continue
    rng = np.random.default_rng(seed=i)
    mz = peaks[:, 0] + rng.normal(0, 0.0003, size=peaks.shape[0])
    ab = np.clip(peaks[:, 1] * (1 + rng.normal(0, 0.02, size=peaks.shape[0])), 0, None)
    name = getattr(row, "compound_name", None) or getattr(row, "name", None) or f"row_{i}"
    sid = getattr(row, "spectra_id", None) or f"spec_{i}"
    pmz = float(getattr(row, "precursormz", 0.0) or 0.0)
    lib_spectra.append(MockSpectrum(mz, ab, name=f"mock_{name}"))
    lib_ids.append(f"mock_{sid}")
    lib_pmz.append(pmz)

print(f"Built {len(lib_spectra)} mock queries from library")

mn = MolecularNetwork(
    fe_lib=fe_lib,
    search_type="open",
    additional_similarities=["cosine"],
    similarity_thresholds={"entropy_similarity": 0.5, "cosine": 0.5},
    use_parallel=False,
)
# All-in-one: query vs library (includes staged computation as implemented)
mn.query_vs_library(
    lib_spectra,
    lib_ids,
    query_precursor_mzs=lib_pmz,
    fe_kwargs=FE_KWARGS,
    hydrate_library_similarities=True,
    library_similarity_threshold=0.3,
)

for metric in ("entropy_similarity", "cosine"):
    edges = mn.get_network_edges(metric=metric)
    stats = mn.get_network_stats(metric=metric)
    print(f"\n[{metric}] n_edges={stats['n_edges']} n_nodes={stats['n_nodes']} density={stats['density']:.3f}")
    for a, b, s in sorted(edges, key=lambda x: -x[2])[:8]:
        print(f"  {a} — {b}: {s:.4f}")


Built 4 mock queries from library
  [query_vs_library] Found 5 query-vs-library pairs above entropy threshold (0.25)
  [query_vs_library] Computing cosine for 4 queries against library...
  [query_vs_library] Stored 5 cosine pairs
  [query_vs_library] Stage 3 – library-vs-library for 5 matched library spectra (threshold=0.3) …
  [query_vs_library] Stage 3 complete – 2 library-vs-library pairs stored.

[entropy_similarity] n_edges=5 n_nodes=9 density=0.139
  mock_CCMSLIB00004684039 — 1: 1.0000
  mock_CCMSLIB00004684071 — mock_CCMSLIB00004684123: 1.0000
  mock_CCMSLIB00004684071 — 2: 1.0000
  mock_CCMSLIB00004684123 — 0: 1.0000
  mock_CCMSLIB00010125334 — 3: 1.0000

[cosine] n_edges=6 n_nodes=9 density=0.167
  mock_CCMSLIB00004684039 — 1: 1.0000
  mock_CCMSLIB00010125334 — 3: 1.0000
  mock_CCMSLIB00004684071 — 2: 1.0000
  mock_CCMSLIB00004684123 — 0: 1.0000
  3 — 4: 0.9999
  mock_CCMSLIB00010125334 — 4: 0.9999


## 4. Export edges and matrices

Write CSV edge lists and a sparse matrix NPZ under `temp.corems/` (gitignored).


In [5]:
edge_csv = OUT_DIR / "tutorial_edges_entropy.csv"
mn.save_edge_list(str(edge_csv), metric="entropy_similarity")
print(f"Wrote {edge_csv}")

mat_csv = OUT_DIR / "tutorial_matrix_entropy.csv"
mn.save_similarity_matrix(str(mat_csv), metric="entropy_similarity", threshold=0.0)
print(f"Wrote {mat_csv}")

npz_path = OUT_DIR / "tutorial_entropy_matrix.npz"
mn.similarity_matrices["entropy_similarity"].save(str(npz_path))
reloaded = SimilarityMatrix.load(str(npz_path))
print(f"Reloaded: {reloaded}")
assert reloaded.n_spectra == mn.similarity_matrices["entropy_similarity"].n_spectra
print("Save/load round-trip OK")


Saved edge list (5 edges) to /Users/heal742/LOCAL/corems_dev/corems/temp.corems/molecular_networking_tutorial/tutorial_edges_entropy.csv
Wrote /Users/heal742/LOCAL/corems_dev/corems/temp.corems/molecular_networking_tutorial/tutorial_edges_entropy.csv
Saved similarity matrix (7 pairs) to /Users/heal742/LOCAL/corems_dev/corems/temp.corems/molecular_networking_tutorial/tutorial_matrix_entropy.csv
Wrote /Users/heal742/LOCAL/corems_dev/corems/temp.corems/molecular_networking_tutorial/tutorial_matrix_entropy.csv
Reloaded: SimilarityMatrix(metric='entropy_similarity', n_spectra=9, n_pairs_stored=7)
Save/load round-trip OK


## 5. Optional visualization

Requires `pip install "corems[networking]"`. If packages are missing, this cell skips cleanly.


In [6]:
try:
    import networkx  # noqa: F401
    import ipysigma  # noqa: F401
    has_viz = True
except ImportError:
    has_viz = False
    print('Optional viz not installed. Run: pip install "corems[networking]"')

if has_viz:
    html_path = OUT_DIR / "tutorial_network_entropy.html"
    mn.plot_network(
        metric="entropy_similarity",
        out_path=str(html_path),
        max_edges=200,
        library_label_field=("compound_name", "name", "spectra_id"),
        bypass_clustering=True,
    )
    print(f"Wrote interactive HTML: {html_path}")
else:
    print("Skipped plot_network")


Optional viz not installed. Run: pip install "corems[networking]"
Skipped plot_network


## 6. Bridge from LC-MS mass features

After you have an `LCMSBase` object with MS2 on mass features (see `LCMS_Tutorial.ipynb`), extract queries with:

```python
from corems.molecular_networking import MolecularNetwork

query_spectra, query_ids, query_precursor_mzs = (
    MolecularNetwork.prepare_query_spectra_from_lcms_object(myLCMSobj)
)
mn = MolecularNetwork(fe_lib=fe_lib, search_type="open", use_parallel=False)
mn.query_vs_library(query_spectra, query_ids, query_precursor_mzs)
```

Only features with a non-empty `best_ms2` are included. Filter with `mf_ids={...}` if needed.


## Summary

| Pattern | API |
|---------|-----|
| Query-only | `MolecularNetwork(fe_lib=None)` + `run_query_vs_query_only` |
| Query + library | `MolecularNetwork(fe_lib=...)` + `query_vs_library` |
| Edges / stats | `get_network_edges`, `get_network_stats` |
| Export | `save_edge_list`, `save_similarity_matrix` |
| LC-MS bridge | `prepare_query_spectra_from_lcms_object` |
| Plot (optional) | `pip install "corems[networking]"` then `plot_network` |

Also see:
- `examples/molecular_networking_demo.py`
- `examples/molecular_networking_queries_only_demo.py`
